# Star Graph Pipeline on Kaggle

Runs enrichment + deep research using a local llama.cpp model (no NVIDIA NIM).
Saves results to /kaggle/working/star-graph/data/. GitHub Actions downloads and pushes back.

Requires the GH_PAT Kaggle Secret; it is read only at runtime.

In [ ]:
# --- clone both repos (GH_PAT from Kaggle Secrets) ---
import base64, subprocess, sys, os, time

for d in ["/kaggle/working/kms", "/kaggle/working/star-graph"]:
    subprocess.run(["rm", "-rf", d], capture_output=True)

from kaggle_secrets import UserSecretsClient
GH_PAT = UserSecretsClient().get_secret("GH_PAT")
if not GH_PAT:
    raise RuntimeError("Attach the GH_PAT secret to this Kaggle notebook")
os.environ["GH_PAT"] = GH_PAT

git_env = os.environ.copy()
auth = base64.b64encode(f"x-access-token:{GH_PAT}".encode()).decode()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})

subprocess.run(["git", "clone", "--depth", "1",
    "https://github.com/Meru143/kaggle-model-server.git", "/kaggle/working/kms"],
    check=True, timeout=120, env=git_env)
subprocess.run(["git", "clone", "--depth", "1",
    "https://github.com/Meru143/star-graph.git", "/kaggle/working/star-graph"],
    check=True, timeout=120, env=git_env)

sys.path.insert(0, "/kaggle/working/kms")
sys.path.insert(0, "/kaggle/working/star-graph/kaggle")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "requests", "networkx", "numpy", "sentence-transformers"],
    check=True, timeout=300)
print("Setup complete")


In [ ]:
# --- imports ------------------------------------------------------------
import importlib
for _m in ('model_registry', 'harness'):
    if _m in sys.modules:
        if _m == 'harness':
            try: sys.modules['harness'].stop()
            except Exception: pass
        importlib.reload(sys.modules[_m])

from model_registry import MODELS
from harness import run, stop, harvest_cache
from star_graph_kaggle import run_pipeline

print(f'Models: {list(MODELS.keys())}')

In [ ]:
# --- boot the local LLM -------------------------------------------------
# First run: downloads GGUF + builds llama.cpp (~15 min)
# Cached run: ~2 min (after harvest_cache done once)

url = run('InternScience/Agents-A1-4B-Q4_K_M-GGUF', MODELS, quant='Q4_K_M', ctx=4096)
print(f'Model ready at {url}')

In [ ]:
# --- run the pipeline (no NVIDIA calls, all local) ----------------------
run_pipeline(limit=None)
print('Pipeline complete. Data saved to /kaggle/working/star-graph/data/')

In [ ]:
# --- cache binaries for faster next boot --------------------------------
try:
    harvest_cache()
    print('Cache harvested')
except Exception as e:
    print(f'Cache harvest skipped: {e}')

In [ ]:
# --- clean shutdown -----------------------------------------------------
stop()
print('Model stopped. Session ending.')